In [1]:

from io import BytesIO
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt
import requests


DATA_URL = (
    "https://raw.githubusercontent.com/"
    "Vadimkin/ukrainian-air-raid-sirens-dataset/"
    "main/datasets/volunteer_data_en.csv"
)

OUTPUT_DIR = Path("outputs")
TEST_DAYS = 28
MIN_HISTORY_DAYS = 60
MAX_DATA_AGE_HOURS = 72


def load_data():
    run_time = pd.Timestamp.now(tz="UTC")

    response = requests.get(DATA_URL, timeout=60)
    response.raise_for_status()

    df = pd.read_csv(BytesIO(response.content))

    required = {"started_at", "region"}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"Missing columns: {missing}. "
            f"Available columns: {list(df.columns)}"
        )

    df["started_at"] = pd.to_datetime(
        df["started_at"], utc=True, errors="coerce"
    )
    df["region"] = df["region"].astype("string").str.strip()

    invalid_time = int(df["started_at"].isna().sum())
    invalid_region = int(
        (df["region"].isna() | df["region"].eq("")).sum()
    )

    df = df.dropna(subset=["started_at", "region"])
    df = df[df["region"] != ""].copy()

    if df.empty:
        raise ValueError("No valid alert records found.")


    latest_event = df["started_at"].max()
    event_age_hours = (
        run_time - latest_event
    ).total_seconds() / 3600

    if event_age_hours > MAX_DATA_AGE_HOURS:
        raise ValueError(
            f"No recent events in the dataset. "
            f"Latest event: {latest_event}; "
            f"event age: {event_age_hours:.1f} hours. "
            "This does not prove that the dataset itself is stale. "
            "Check the source and its update history."
        )

    metadata = {
        "source_url": DATA_URL,
        "run_time_utc": str(run_time),
        "latest_event_timestamp_utc": str(latest_event),
        "downloaded_rows": int(len(df)),
        "invalid_timestamps_removed": invalid_time,
        "invalid_regions_removed": invalid_region,
        "data_age_hours": round(
            (run_time - latest_event).total_seconds() / 3600, 2
        ),
        "note": (
            "Latest event timestamp measures event recency, "
            "not dataset completeness."
        ),
    }

    print("Rows:", len(df))
    print("Latest event:", latest_event)
    print("Data age (hours):", metadata["data_age_hours"])

    return df, metadata


def build_daily_series(df):
    df = df.copy()
    df["date"] = df["started_at"].dt.floor("D")

    daily = df.groupby("date").size().sort_index()

    full_index = pd.date_range(
        start=daily.index.min(),
        end=daily.index.max(),
        freq="D",
        tz="UTC",
    )

    missing_dates = full_index.difference(daily.index)

    # Assumption: a date without rows means zero recorded events.
    # The source does not independently prove completeness for
    # every such date, so we report this assumption explicitly.
    daily = daily.reindex(full_index, fill_value=0)
    daily.index.name = "date"
    daily.name = "alert_starts"

    return daily, len(missing_dates)


def evaluate_forecasts(daily):
    run_time = pd.Timestamp.now(tz="UTC")
    today_utc = run_time.normalize()
    last_complete_date = today_utc - pd.Timedelta(days=1)

    # Exclude the current, potentially incomplete UTC day.
    daily = daily.loc[:last_complete_date].copy()

    if daily.empty:
        raise ValueError("No complete days available.")

    if len(daily) < MIN_HISTORY_DAYS + TEST_DAYS:
        raise ValueError(
            f"Need at least {MIN_HISTORY_DAYS + TEST_DAYS} "
            f"daily observations; found {len(daily)}."
        )

    test_start = len(daily) - TEST_DAYS
    rows = []

    for i in range(test_start, len(daily)):
        history = daily.iloc[:i]
        actual = int(daily.iloc[i])

        rows.append({
            "date": daily.index[i],
            "actual": actual,
            "previous_day": float(history.iloc[-1]),
            "previous_7_day_mean": float(history.iloc[-7:].mean()),
        })

    results = pd.DataFrame(rows).set_index("date")

    metrics = []
    for model in ["previous_day", "previous_7_day_mean"]:
        errors = (results["actual"] - results[model]).abs()
        metrics.append({
            "model": model,
            "MAE": float(errors.mean()),
            "test_days": len(results),
        })

    metrics_df = pd.DataFrame(metrics)

    return daily, results, metrics_df


def save_outputs(daily, results, metrics, metadata, zero_days):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    results.to_csv(OUTPUT_DIR / "test_predictions.csv")
    metrics.to_csv(OUTPUT_DIR / "metrics.csv", index=False)

    # Save the exact daily series used for evaluation.
    daily.rename("alert_starts").to_csv(
        OUTPUT_DIR / "daily_series.csv"
    )

    metadata["zero_filled_dates"] = zero_days
    metadata["evaluation_start_utc"] = str(results.index.min())
    metadata["evaluation_end_utc"] = str(results.index.max())
    metadata["test_days"] = len(results)
    metadata["target"] = (
        "Number of recorded regional alert starts per UTC day"
    )
    metadata["validation"] = (
        "Rolling one-day-ahead evaluation; each forecast uses "
        "only observations preceding its target date."
    )

    with open(OUTPUT_DIR / "run_metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(results.index, results["actual"], label="Actual", marker="o")
    ax.plot(
        results.index, results["previous_day"],
        label="Previous-day baseline"
    )
    ax.plot(
        results.index, results["previous_7_day_mean"],
        label="Previous-7-day mean"
    )

    ax.set_title("Daily recorded air-alert starts (UTC)")
    ax.set_xlabel("Date")
    ax.set_ylabel("Recorded alert starts")
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "forecast_evaluation.png", dpi=150)
    plt.close(fig)

    # One-day-ahead forecast from the last complete day.
    # Its target may be the current day, not tomorrow.
    origin_date = daily.index[-1]
    target_date = origin_date + pd.Timedelta(days=1)

    forecast = pd.DataFrame([{
        "forecast_origin_date_utc": str(origin_date.date()),
        "forecast_target_date_utc": str(target_date.date()),
        "previous_day_baseline": float(daily.iloc[-1]),
        "previous_7_day_mean": float(daily.iloc[-7:].mean()),
    }])

    forecast.to_csv(
        OUTPUT_DIR / "one_day_ahead_forecast.csv", index=False
    )

    print("\nTest period:", results.index.min().date(),
          "to", results.index.max().date())
    print("\nMetrics (lower MAE is better):")
    print(metrics.to_string(index=False))
    print("\nDates filled with zero:", zero_days)
    print("Outputs:", OUTPUT_DIR.resolve())


def main():
    df, metadata = load_data()
    daily, zero_days = build_daily_series(df)
    daily, results, metrics = evaluate_forecasts(daily)
    save_outputs(daily, results, metrics, metadata, zero_days)


if __name__ == "__main__":
    main()


Rows: 116174
Latest event: 2026-10-12 04:49:48+00:00
Data age (hours): 15.37

Test period: 2026-09-14 to 2026-10-11

Metrics (lower MAE is better):
              model       MAE  test_days
       previous_day 12.607143         28
previous_7_day_mean 18.357143         28

Dates filled with zero: 0
Outputs: /content/outputs
